In [2]:
import os
import re
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [3]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class TextClassifierDNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(TextClassifierDNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size) 
        self.relu = nn.ReLU() 
        self.dropout = nn.Dropout(0.2) 
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out) 
        return out

def load_pytorch_model(input_size, hidden_size, n_classes, filepath):
    model = TextClassifierDNN(input_size, hidden_size, n_classes)
    model.load_state_dict(torch.load(filepath, map_location="cpu"))
    model.eval()
    print(f"Modelo PyTorch carregado de {filepath}")
    return model

def predict_pytorch(model, X_test, n_classes):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32)
        outputs = model(X_tensor)

        if n_classes == 1:
            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).int()
        else:
            _, predicted = torch.max(outputs.data, 1)

    return predicted.numpy()

In [4]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "at", "for", "with",
    "is", "are", "was", "were", "be", "been", "being", "this", "that", "these",
    "those", "it", "its", "as", "by", "from", "but", "about", "into", "than",
    "then", "so", "such", "if", "their", "there", "they", "them", "he", "she",
    "you", "your", "we", "our", "i", "my", "me", "his", "her", "what", "which",
    "who", "whom", "can", "could", "should", "would", "do", "does", "did", "have",
    "has", "had", "not", "no", "yes", "will", "just"
}

def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b[a-zA-ZÀ-ÿ]{2,}\b", text)
    return [tok for tok in tokens if tok not in STOPWORDS]

def vectorize_tfidf(texts, vocab, idf):
    X = np.zeros((len(texts), len(vocab)), dtype=np.float64)

    for i, text in enumerate(texts):
        tokens = [tok for tok in tokenize(text) if tok in vocab]
        if not tokens:
            continue

        counts = Counter(tokens)
        total_terms = len(tokens)

        for tok, count in counts.items():
            j = vocab[tok]
            tf = count / total_terms
            X[i, j] = tf * idf[j]

    return X

def l2_normalize_rows(X, eps=1e-12):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (norms + eps)

In [11]:
GROUP_NUMBER = 4
COURSE = "MIA"

SUBM1_INPUT_PATH = "subm1.csv"
OUTPUT_CSV_PATH = f"subm1-g{GROUP_NUMBER}-{COURSE}-B.csv"

BINARY_MODEL_PATH = "pytorch_binary_model.pt"
LLM_MODEL_PATH = "pytorch_multiclass_llm_model.pt"

print("Output:", OUTPUT_CSV_PATH)

Output: subm1-g4-MIA-B.csv


In [12]:
with open("tfidf_artifacts.pkl", "rb") as f:
    data = pickle.load(f)

vocab = data["vocab"]
idf = data["idf"]

print("TF-IDF carregado com sucesso.")
print("Vocab size:", len(vocab))

TF-IDF carregado com sucesso.
Vocab size: 5000


In [13]:
input_dim = len(vocab)

model_binary = load_pytorch_model(
    input_size=input_dim,
    hidden_size=64,
    n_classes=1,
    filepath=BINARY_MODEL_PATH
)

model_llm = load_pytorch_model(
    input_size=input_dim,
    hidden_size=64,
    n_classes=4,
    filepath=LLM_MODEL_PATH
)

Modelo PyTorch carregado de pytorch_binary_model.pt
Modelo PyTorch carregado de pytorch_multiclass_llm_model.pt


In [14]:
df_subm1 = pd.read_csv(SUBM1_INPUT_PATH, sep=None, engine="python")
df_subm1.columns = df_subm1.columns.str.replace('\ufeff', '')

print(df_subm1.shape)
df_subm1.head()

(150, 2)


,ID,Text
0,D2-1,A covalent bond is a chemical bond that involv...
1,D2-2,A covalent bond forms when two atoms share one...
2,D2-3,A covalent bond is a type of chemical bond whe...
3,D2-4,A covalent bond is a chemical bond that involv...
4,D2-5,Driven by exciting developments in the field o...


In [16]:
X_subm1_text = df_subm1["Text"].fillna("").astype(str).values
X_subm1 = vectorize_tfidf(X_subm1_text, vocab, idf)
X_subm1 = l2_normalize_rows(X_subm1)

print("X_subm1 shape:", X_subm1.shape)

X_subm1 shape: (150, 5000)


In [17]:
pred_bin = predict_pytorch(model_binary, X_subm1, n_classes=1).flatten()

print("Humans:", int((pred_bin == 0).sum()))
print("LLMs:", int((pred_bin == 1).sum()))

Humans: 74
LLMs: 76


In [18]:
indices_llm = np.where(pred_bin == 1)[0]
X_subm1_llm = X_subm1[indices_llm]

if len(indices_llm) > 0:
    pred_llm = predict_pytorch(model_llm, X_subm1_llm, n_classes=4)
else:
    pred_llm = np.array([], dtype=int)

print("Número de exemplos previstos como LLM:", len(indices_llm))

Número de exemplos previstos como LLM: 76


In [19]:
llm_mapping = {
    0: "OpenAI",
    1: "Google",
    2: "Meta",
    3: "Anthropic"
}

final_labels = np.array(["Human"] * len(df_subm1), dtype=object)

for pos, pred in zip(indices_llm, pred_llm):
    final_labels[pos] = llm_mapping[int(pred)]

print(pd.Series(final_labels).value_counts())

Human        74
Google       35
Anthropic    27
OpenAI       13
Meta          1
Name: count, dtype: int64


In [20]:
df_output = df_subm1.copy()
df_output.columns = df_output.columns.str.replace('\ufeff', '')
df_output["Labels"] = final_labels

df_output.to_csv(OUTPUT_CSV_PATH, index=False)

print("CSV criado em:", OUTPUT_CSV_PATH)
df_output.head()

CSV criado em: subm1-g4-MIA-B.csv


,ID,Text,Labels
0,D2-1,A covalent bond is a chemical bond that involv...,Human
1,D2-2,A covalent bond forms when two atoms share one...,Human
2,D2-3,A covalent bond is a type of chemical bond whe...,Human
3,D2-4,A covalent bond is a chemical bond that involv...,OpenAI
4,D2-5,Driven by exciting developments in the field o...,Human


In [21]:
df_check = pd.read_csv(OUTPUT_CSV_PATH)

print(df_check.shape)
print(df_check["Labels"].value_counts())
df_check.head()

(150, 3)
Labels
Human        74
Google       35
Anthropic    27
OpenAI       13
Meta          1
Name: count, dtype: int64


,ID,Text,Labels
0,D2-1,A covalent bond is a chemical bond that involv...,Human
1,D2-2,A covalent bond forms when two atoms share one...,Human
2,D2-3,A covalent bond is a type of chemical bond whe...,Human
3,D2-4,A covalent bond is a chemical bond that involv...,OpenAI
4,D2-5,Driven by exciting developments in the field o...,Human
